# ApuCast v3 en Kaggle con dos GPU T4

El candidato combina la CNN v2 archivada y un ensamble ResNet con pesos de mezcla distintos para RH, T, U y V. Este notebook reentrena las tres ResNet y vuelve a exportar el CSV con la calibraci?n versionada. Para subir el archivo publicado con el hash registrado, usa `versions/v3/submission_v3.csv`.

Adjunta la competencia **Hackathon IA PUCP 2026** y selecciona **GPU T4 x2**. Si usas VS Code, elige el servidor de Kaggle como kernel. Habilita Internet si todav?a necesitas clonar la rama `production`.

## 1. Configuraci?n y verificaci?n

Usa `PRUEBA_RAPIDA = True` para verificar las dos GPU con dos semillas y una ?poca. En modo completo, la nueva exportaci?n queda en `/kaggle/working/submission_v3.csv`.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import subprocess
import sys

import numpy as np
import pandas as pd
import torch

PRUEBA_RAPIDA = False
RAIZ = Path("/kaggle/working")
REPO = RAIZ / "ApuCast-production"
RAMA = "production"
SEMILLAS = (0, 1) if PRUEBA_RAPIDA else (0, 1, 2)
EPOCAS = 1 if PRUEBA_RAPIDA else 25
SALIDAS = RAIZ / ("apucast_v3_prueba" if PRUEBA_RAPIDA else "apucast_v3_ejecucion")
TRABAJO = SALIDAS / "resnet"

if torch.cuda.device_count() != 2:
    raise RuntimeError("Selecciona GPU T4 x2 en Kaggle y vuelve a ejecutar la celda.")

for index in range(2):
    print(f"GPU {index}: {torch.cuda.get_device_name(index)}")

## 2. C?digo y datos oficiales

Usa la entrada oficial de Kaggle para una sesi?n nueva. La rama `production` re?ne el c?digo, la calibraci?n y las versiones de ApuCast.

In [ ]:
if not REPO.exists():
    subprocess.run(
        ["git", "clone", "--single-branch", "--branch", RAMA,
         "https://github.com/ema28pro/ApuCast-Hack-PUCP.git", str(REPO)],
        check=True,
    )

head = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
print(f"Commit de ApuCast: {head}")

archivos = ("train.npz", "test.npz", "static.npz", "sample_submission.csv")
entradas = Path("/kaggle/input")
candidatos = [
    p.parent for p in entradas.rglob("train.npz")
    if "hackathon-ia-pucp-2026" in p.parts
]
candidatos.extend([REPO / "data", RAIZ / "ApuCast-Hack-PUCP" / "data"])
DATOS = next((p for p in candidatos if all((p / f).is_file() for f in archivos)), None)
if DATOS is None:
    raise FileNotFoundError("Adjunta los cuatro archivos oficiales desde Input > Add Input.")
print(f"Datos: {DATOS}")

TRABAJO.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(REPO / "train"))

## 3. Entrena las ResNet en paralelo

La primera T4 entrena las semillas 0 y 2. La segunda entrena la 1. Cada proceso utiliza la arquitectura, el optimizador y el protocolo de la rama v2.1. Guarda un checkpoint por semilla terminada y muestra el avance por época. Si vuelves a ejecutar esta celda en la misma sesión, reutiliza los checkpoints del mismo modo de ejecución.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

worker = r"""
import sys
from pathlib import Path

import numpy as np

from generar_submission_v3 import predict, train_seed
from runtime_apucast import load_starter

repo, data, work = map(Path, sys.argv[1:4])
epochs = int(sys.argv[4])
seeds = [int(value) for value in sys.argv[5:]]
ns = load_starter(repo, data)
hour, tensor = ns["hour"], ns["to_t"]
train_hours = hour(ns["ttr"])
test_hours = hour(ns["tte"])
clim = ns["climatology"](ns["Ytr"], ns["ttr"])
mu = ns["Xtr"].mean(axis=(0, 1, 2))
sd = ns["Xtr"].std(axis=(0, 1, 2))
anomalies = ns["Ytr"] - clim[train_hours]
scale = anomalies.std(axis=(0, 1, 2))
train = tensor((ns["Wtr"] - mu) / sd)
targets = tensor(anomalies / scale)
test = tensor((ns["Wte"] - mu) / sd)
train_hours = tensor(train_hours)
test_reference = tensor(clim[test_hours])
test_hours = tensor(test_hours)
scale_tensor = tensor(scale)

if 0 in seeds:
    np.savez_compressed(work / "preprocesamiento.npz", mu=mu, sd=sd, scale=scale, climatology=clim)
    np.save(work / "horas_test.npy", ns["tte"], allow_pickle=False)

for seed in seeds:
    model = train_seed(ns, seed, epochs, train, train_hours, targets, work / f"resnet_{seed}.pt")
    prediction = predict(model, test, test_hours, test_reference, scale_tensor)
    np.save(work / f"prediccion_{seed}.npy", prediction, allow_pickle=False)
    del model
"""

def entrenar_gpu(dispositivo, semillas):
    visibles = os.environ.get("CUDA_VISIBLE_DEVICES", "0,1").split(",")
    env = {
        **os.environ,
        "CUDA_VISIBLE_DEVICES": visibles[dispositivo],
        "PYTHONPATH": str(REPO / "train"),
        "OMP_NUM_THREADS": "2",
        "MKL_NUM_THREADS": "2",
    }
    comando = [sys.executable, "-u", "-c", worker, str(REPO), str(DATOS), str(TRABAJO),
               str(EPOCAS), *map(str, semillas)]
    print(f"GPU {dispositivo}: semillas {semillas}", flush=True)
    with subprocess.Popen(comando, cwd=REPO, env=env, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as proceso:
        for linea in proceso.stdout:
            print(f"[GPU {dispositivo}] {linea.rstrip()}", flush=True)
        if proceso.wait():
            raise subprocess.CalledProcessError(proceso.returncode, f"Entrenamiento GPU {dispositivo}")

with ThreadPoolExecutor(max_workers=2) as pool:
    tareas = [pool.submit(entrenar_gpu, gpu, SEMILLAS[gpu::2]) for gpu in range(2)]
    for tarea in tareas:
        tarea.result()

## 4. Exporta y valida el CSV v3

La exportaci?n verifica los hashes de ambos CSV de entrada, aplica los pesos versionados y valida el orden de IDs frente a la plantilla. La prueba r?pida solo exporta el componente ResNet de dos semillas.

In [ ]:
from backtest_mezcla import export_calibrated
from generar_mezcla_v3 import check, read_verified
from generar_submission_v3 import VARIABLES, save_submission

plantilla = pd.read_csv(DATOS / "sample_submission.csv")
horas = np.load(TRABAJO / "horas_test.npy", allow_pickle=False)
predicciones = [np.load(TRABAJO / f"prediccion_{s}.npy", allow_pickle=False) for s in SEMILLAS]
pronostico = np.mean(predicciones, axis=0, dtype=np.float32)
pronostico[..., 0] = pronostico[..., 0].clip(0, 100)

resnet_csv = SALIDAS / "resnet.csv"
resnet_sha = save_submission(plantilla, pronostico, horas, resnet_csv)
resnet_csv.with_suffix(".json").write_text(json.dumps({
    "archivo": resnet_csv.name,
    "sha256": resnet_sha,
    "semillas": list(SEMILLAS),
    "epocas_fijas": EPOCAS,
    "prueba_rapida": PRUEBA_RAPIDA,
    "score_publico": None,
}, indent=2) + "\n", encoding="utf-8")

def ejecutar_script(script, *argumentos):
    subprocess.run([sys.executable, str(REPO / "train" / script), *map(str, argumentos)],
                   cwd=REPO, check=True)

if PRUEBA_RAPIDA:
    print(f"Prueba rápida completada: {resnet_csv}")
else:
    v2_csv = SALIDAS / "v2_top.csv"
    v2_json = v2_csv.with_suffix(".json")
    if not v2_csv.exists() and not v2_json.exists():
        ejecutar_script("exportar_v2.py", "--data", DATOS, "--output", v2_csv)

    release = REPO / "versions/v3/submission_v3.json"
    pesos = json.loads(release.read_text(encoding="utf-8"))["pesos_resnet"]
    weights = np.array([pesos[name] for name in VARIABLES])
    final_csv = RAIZ / "submission_v3.csv"
    final_json = final_csv.with_suffix(".json")
    if not final_csv.exists() and not final_json.exists():
        export_calibrated(DATOS, SALIDAS, weights, final_csv)

    final, sha, registro = read_verified(final_csv, final_json)
    check(final, plantilla)
    _, v2_sha, _ = read_verified(v2_csv, v2_json)
    if registro["fuentes_sha256"] != {"v2": v2_sha, "resnet": resnet_sha}:
        raise RuntimeError("El CSV existente corresponde a otra ejecuci?n. Guarda el CSV y su JSON en otra carpeta antes de volver a exportar.")
    print(f"Env?o listo: {final_csv} | {len(final):,} filas | SHA-256 {sha}")


## 5. Entrega

El CSV publicado de v3 tiene un score retrospectivo de **0,421261** en tres cortes y tres semillas. La evaluaci?n p?blica de Kaggle queda pendiente. Al regenerar el archivo, conserva su manifiesto para registrar el SHA-256 exacto evaluado por Kaggle.